In [ ]:
!pip install gdown

In [ ]:
#start spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HR Attrition") \
        .getOrCreate()

In [ ]:
#masukkan csv -- disini pakai gdown
#download lgdg dari link drive pakai ID
import gdown

file_id = "1FLaBeDwTdijHmxnLPmMY1ssudwlk58YZ"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.csv", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1FLaBeDwTdijHmxnLPmMY1ssudwlk58YZ
To: /content/data.csv
100%|██████████| 227k/227k [00:00<00:00, 66.4MB/s]


'data.csv'

In [ ]:
#load data ke spark
df = spark.read.csv("data.csv", header=True, inferSchema=True)
df.show(5)

+---+---------+-----------------+---------+--------------------+----------------+---------+--------------+-------------+--------------+-----------------------+------+----------+--------------+--------+--------------------+---------------+-------------+-------------+-----------+------------------+------+--------+-----------------+-----------------+------------------------+-------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+
|Age|Attrition|   BusinessTravel|DailyRate|          Department|DistanceFromHome|Education|EducationField|EmployeeCount|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|             JobRole|JobSatisfaction|MaritalStatus|MonthlyIncome|MonthlyRate|NumCompaniesWorked|Over18|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StandardHours|StockOptionLevel|TotalWorkingYears|TrainingTimesLastYear|WorkLifeBalanc

In [ ]:
#DATA CLEANING -- fix BOM
df = df.toDF(*[c.replace("ï»¿", "") for c in df.columns])

In [ ]:
#DATA CLEANING -- rename target
from pyspark.sql.functions import col, when

df = df.withColumnRenamed("Attrition", "left_company")

In [ ]:
#DATA CLEANING -- convert target ke numeric
from pyspark.sql.functions import col, when

df = df.withColumn(
    "left_company",
        when(col("left_company") == "Yes", 1).otherwise(0)
        )

In [ ]:
#DATA CLEANING -- drop kolom tidak penting
df = df.drop("EmployeeCount", "StandardHours", "Over18", "EmployeeNumber")

In [ ]:
#DATA CLEANING -- filter data kotor
from pyspark.sql.functions import col

df = df.filter(col("Age").rlike("^[0-9]+$"))

In [ ]:
# casting
from pyspark.sql.functions import col

df = df.withColumn("Age", col("Age").cast("int")) \
       .withColumn("MonthlyIncome", col("MonthlyIncome").cast("int")) \
       .withColumn("JobSatisfaction", col("JobSatisfaction").cast("int")) \
       .withColumn("PerformanceRating", col("PerformanceRating").cast("int")) \
       .withColumn("WorkLifeBalance", col("WorkLifeBalance").cast("int")) \
       .withColumn("YearsAtCompany", col("YearsAtCompany").cast("int"))

In [ ]:
#FIX DATA TYPE
from pyspark.sql.functions import col

df = df.withColumn("Age", col("Age").cast("int")) \
       .withColumn("MonthlyIncome", col("MonthlyIncome").cast("int")) \
       .withColumn("JobSatisfaction", col("JobSatisfaction").cast("int")) \
       .withColumn("PerformanceRating", col("PerformanceRating").cast("int")) \
       .withColumn("WorkLifeBalance", col("WorkLifeBalance").cast("int")) \
       .withColumn("YearsAtCompany", col("YearsAtCompany").cast("int"))

In [ ]:
#FEATURE ENGINEERING -- encoding kategori
from pyspark.ml.feature import StringIndexer

categorical_cols = ["Department", "JobRole", "Gender", "MaritalStatus", "OverTime"]

for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_index")
    df = indexer.fit(df).transform(df)

In [ ]:
#cek
df.select("Department", "Department_index").show(5)

+--------------------+----------------+
|          Department|Department_index|
+--------------------+----------------+
|               Sales|             1.0|
|Research & Develo...|             0.0|
|Research & Develo...|             0.0|
|Research & Develo...|             0.0|
|Research & Develo...|             0.0|
+--------------------+----------------+
only showing top 5 rows


In [ ]:
#FEATURE ENGINEERING -- gabung fitur numerik
from pyspark.ml.feature import VectorAssembler

numeric_cols = [
    "Age", "MonthlyIncome", "JobSatisfaction",
    "PerformanceRating", "WorkLifeBalance",
    "YearsAtCompany"
]

assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="num_features"
)

df = assembler.transform(df)

In [ ]:
#cek
df.select("num_features").show(5)

+--------------------+
|        num_features|
+--------------------+
|[41.0,5993.0,4.0,...|
|[49.0,5130.0,2.0,...|
|[37.0,2090.0,3.0,...|
|[33.0,2909.0,3.0,...|
|[27.0,3468.0,2.0,...|
+--------------------+
only showing top 5 rows


In [ ]:
#FEATURE ENGINEERING -- scaling
from pyspark.ml.feature import MinMaxScaler

scaler = MinMaxScaler(inputCol="num_features", outputCol="scaled_features")
df = scaler.fit(df).transform(df)

In [ ]:
#cek
df.select("scaled_features").show(5)

+--------------------+
|     scaled_features|
+--------------------+
|[0.54761904761904...|
|[0.73809523809523...|
|[0.45238095238095...|
|[0.35714285714285...|
|[0.21428571428571...|
+--------------------+
only showing top 5 rows


In [ ]:
#SPLIT DATA
train, test = df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
#FINAL DATASET
df_final = df.select(
    "scaled_features",
    "Department_index",
    "JobRole_index",
    "Gender_index",
    "OverTime_index",
    "left_company"
)

df_final.show(5)

+--------------------+----------------+-------------+------------+--------------+------------+
|     scaled_features|Department_index|JobRole_index|Gender_index|OverTime_index|left_company|
+--------------------+----------------+-------------+------------+--------------+------------+
|[0.54761904761904...|             1.0|          0.0|         1.0|           1.0|           1|
|[0.73809523809523...|             0.0|          1.0|         0.0|           0.0|           0|
|[0.45238095238095...|             0.0|          2.0|         0.0|           1.0|           1|
|[0.35714285714285...|             0.0|          1.0|         1.0|           1.0|           0|
|[0.21428571428571...|             0.0|          2.0|         0.0|           0.0|           0|
+--------------------+----------------+-------------+------------+--------------+------------+
only showing top 5 rows
